In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean
Configuration: {'general': {'run_name': 'experiment_with_13_classes', 'seed': 42, 'n_classes': 13}, 'dataset': {'split_type': 'test', 'test_split': 0.2, 'cutoff_year': 1996}, 'paths': {'data_exploration_dir': 'experiment_with_13_classes/data_exploration', 'artifacts_dir': 'experiment_with_13_classes/artifacts', 'embeddings_dir': 'experiment_with_13_classes/embeddings', 'models_dir': 'experiment_with_13_classes/models', 'results_dir': 'experiment_with_13_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_CUTOFF_YEAR: 1996
  DATASET_SPLIT_TYP

# 06 – Semantic Search Demo

Interactive semantic search on the Reuters documents using a MiniLM embedding model. Optionally, this notebook can switch to a Retrieval‑Augmented Generation (RAG) mode that synthesises an answer from the top retrieved documents.

In [2]:

import os, pathlib, pickle, faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from src.datasets.dataset import get_dataset

EMB_DIR = pathlib.Path('embeddings')
EMB_DIR.mkdir(exist_ok=True, parents=True)
INDEX_PATH = EMB_DIR / 'minilm.index'
DOCS_PATH  = EMB_DIR / 'docs.pkl'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.
/home/marcmaceira/projects/reuters-rag-classifier_clean/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load documents (train + test for demo)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    cutoff_year=GENERAL_CUTOFF_YEAR
)

INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 13
INFO |   - Samples per class: 20
INFO |   - Random seed: None
INFO | Loading small test dataset with 20 samples per class across 13 classes
INFO | Selected classes: earn, acq, crude, interest, money-fx, trade, grain, corn, dlr, money-supply, ship, coffee, sugar
INFO |   - Class 'earn': 14 train, 6 test
INFO |   - Class 'acq': 14 train, 6 test
INFO |   - Class 'crude': 14 train, 6 test
INFO |   - Class 'interest': 14 train, 6 test
INFO |   - Class 'money-fx': 14 train, 6 test
INFO |   - Class 'trade': 14 train, 6 test
INFO |   - Class 'grain': 14 train, 6 test
INFO |   - Class 'corn': 14 train, 6 test
INFO |   - Class 'dlr': 14 train, 6 test
INFO |   - Class 'money-supply': 14 train, 6 test
INFO |   - Class 'ship': 14 train, 6 test
INFO |   - Class 'coffee': 14 train, 6 test
INFO |   - Class 'sugar': 14 train, 6 test


In [4]:



docs = X_train + X_test
if not INDEX_PATH.exists():
    print('Building embeddings…')
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(docs, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
    dimension = emb.shape[1]
    index = faiss.IndexFlatIP(dimension)
    # normalise for cosine sim
    faiss.normalize_L2(emb)
    index.add(emb)
    faiss.write_index(index, str(INDEX_PATH))
    with open(DOCS_PATH, 'wb') as f:
        pickle.dump(docs, f)
    print(f'Index and docs saved to {EMB_DIR}')
else:
    print('Embeddings already built. Loading…')
    index = faiss.read_index(str(INDEX_PATH))
    with open(DOCS_PATH, 'rb') as f:
        docs = pickle.load(f)
    model = SentenceTransformer(MODEL_NAME)


INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Embeddings already built. Loading…


In [5]:

# ---- Search helper ----
def search(query: str, k: int = 5):
    q_emb = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    print(f"Top {k} results:")
    for rank, (idx, score) in enumerate(zip(I[0], D[0]), 1):
        print(f"{rank}. (score={score:.3f}) {docs[idx][:200]}…\n")


In [6]:

# Demo
search("oil prices in saudi arabia")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 44.31it/s]

Top 5 results:
1. (score=0.486) RECENT U.S. OIL DEMAND UP 0.1 PCT FROM YEAR AGO
  U.S. oil demand as measured by
  products supplied rose 0.1 pct in the four weeks ended March 20
  to 16.16 mln barrels per day from 16.15 mln in the …

2. (score=0.465) KUWAIT SAYS OPEC 2.4 MLN BPD BELOW CEILING
  Kuwaiti oil minister Sheikh Ali
  al-Khalifa al-Sabah said OPEC was producing well below its oil
  output ceiling and this would help prices move higher,
 …

3. (score=0.450) VENEZUELA SEES OIL STABILITY DESPITE GULF ATTACK
  Venezuelan Energy Minister Arturo
  Hernandez Grisanti said he foresaw market stability in the
  price of crude, despite growing tension in the Gulf …

4. (score=0.442) SUPPLIES, MIDEAST TENSION FUEL GAINS IN OIL
  Petroleum futures rallied today in a
  market that was expecting declines in domestic supplies and
  became further unsettled by escalated Mideast fightin…

5. (score=0.418) SOUTHLAND &lt;SLC> UNIT RAISES CRUDE PRICES
  Southland Corp's Citgo Petrleum Corp
  sai

### Optional – Retrieval‑Augmented Generation (RAG)
If you have an OpenAI key configured, you can uncomment the cell below to generate answers from the retrieved passages.

In [7]:
# !pip install openai
from openai import OpenAI
import textwrap
import os

# Initialize the OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY env var not set.')

def rag_answer(question: str, k: int = 5) -> str:
    q_emb = model.encode([question], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, k)
    
    # Collect reference documents
    references = []
    for rank, (idx, score) in enumerate(zip(I[0], D[0]), 1):
        references.append(f"Document {rank} (score={score:.3f}): {docs[idx]}")
    
    # Join all reference documents for context
    context = "\n".join([docs[i] for i in I[0]])
    prompt = f"Answer the question based only on the context below.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    
    # Generate answer
    response = client.completions.create(
        model='gpt-3.5-turbo-instruct',
        prompt=prompt,
        max_tokens=256
    )
    answer = textwrap.dedent(response.choices[0].text).strip()
    
    # Return both the answer and the reference documents
    full_response = f"Answer:\n{answer}\n\nReference Documents:\n" + "\n\n".join(references)
    return full_response

# Example - uncomment to test
print(rag_answer("What is the effects of the Regan administration?"))

Batches: 100%|██████████| 1/1 [00:00<00:00, 68.44it/s]
INFO | HTTP Request: POST https://api.openai.com/v1/completions "HTTP/1.1 200 OK"


Answer:
The effects of the Reagan administration included higher economic growth, a reduction in inflation, increased military spending and a growing trade deficit. They also implemented policies of deregulation and tax cuts, which led to a widening income gap between the rich and poor. There were also negative effects such as increased government debt and a decrease in social programs.

Reference Documents:
Document 1 (score=0.345): MULFORD SAYS G-6 WANTS STABILITY
  Treasury Assistant Secretary David
  Mulford said the Paris agreement among leading industrial
  nations is intended to produce "reasonable stability" in exchange
  markets over the next few months.
      He told a Senate Banking subcommittee the Group of Five
  nations and Canada agreed in Paris to "see if there can't be a
  period of reasonable stability instead of volatility" to give
  time for the committments in Paris to take place.
      Asked by Sen Phil Gramm (R-Tex) whether U.S. intervention
  was not in fact ove